In [1]:
import sys
import cudaq

/tmp/ipykernel_463/2062711558.py:2: FutureWarning: The CUDA-Q `sample` and `observe` algorithmic primitives will change in a future release. Existing code may require updates. See https://nvidia.github.io/cuda-quantum/latest/using/migration/upcoming_changes.html for details.
  import cudaq


### SAMPLE

In [8]:
print(f"Running on target {cudaq.get_target().name}")
qubit_count = 4

Running on target qpp-cpu


In [15]:
@cudaq.kernel
def kernel(qubit_count: int):
    qubits = cudaq.qvector(qubit_count)
    h(qubits[0])
    for i in range(1, qubit_count):
        x.ctrl(qubits[0], qubits[i])
    mz(qubits)

In [18]:
qubit_count = 4
print(cudaq.draw(kernel, qubit_count))

     ╭───╮               
q0 : ┤ h ├──●────●────●──
     ╰───╯╭─┴─╮  │    │  
q1 : ─────┤ x ├──┼────┼──
          ╰───╯╭─┴─╮  │  
q2 : ──────────┤ x ├──┼──
               ╰───╯╭─┴─╮
q3 : ───────────────┤ x ├
                    ╰───╯



In [20]:
results = cudaq.sample(kernel, qubit_count,shots_count=1000)
# Should see a roughly 50/50 distribution between the |00> and
# |11> states. Example: {00: 505  11: 495}
print("Measurement distribution:" + str(results))

Measurement distribution:{ 0000:484 1111:516 }



In [21]:
most_probable_result = results.most_probable()
probability = results.probability(most_probable_result)
print("Most probable result: " + most_probable_result)
print("Measured with probability " + str(probability), end='\n\n')

Most probable result: 1111
Measured with probability 0.516



### RUN

In [22]:
# Define a quantum kernel that returns an integer
@cudaq.kernel
def ghz_kernel(qubit_count: int) -> int:
    # Allocate qubits
    qubits = cudaq.qvector(qubit_count)

    # Create GHZ state
    h(qubits[0])
    for i in range(1, qubit_count):
        x.ctrl(qubits[0], qubits[i])

    # Measure and count the number of qubits in state |1⟩
    result = 0
    for i in range(qubit_count):
        if mz(qubits[i]):
            result += 1

    return result

In [23]:
# Execute the kernel multiple times and collect individual results
qubit_count = 3
results = cudaq.run(ghz_kernel, qubit_count, shots_count=10)
print(f"Executed {len(results)} shots")
print(f"Results: {results}")

Executed 10 shots
Results: [3, 0, 3, 0, 0, 0, 0, 0, 3, 0]


In [24]:
# Count occurrences of each result
value_counts = {}
for value in results:
    value_counts[value] = value_counts.get(value, 0) + 1

print("\nCounts of each result:")
for value, count in sorted(value_counts.items()):
    print(f"Result {value}: {count} times")

# Analyze patterns in the results
zero_count = results.count(0)
full_count = results.count(qubit_count)
other_count = len(results) - zero_count - full_count
print(f"\nGHZ state analysis:")
print(
    f"  All qubits in |0⟩: {zero_count} times ({zero_count/len(results)*100:.1f}%)"
)
print(
    f"  All qubits in |1⟩: {full_count} times ({full_count/len(results)*100:.1f}%)"
)
print(
    f"  Other states: {other_count} times ({other_count/len(results)*100:.1f}%)"
)


Counts of each result:
Result 0: 7 times
Result 3: 3 times

GHZ state analysis:
  All qubits in |0⟩: 7 times (70.0%)
  All qubits in |1⟩: 3 times (30.0%)
  Other states: 0 times (0.0%)


### OBSERVE
The `observe` function allows us to calculate expectation values for a defined quantum operator, that is the value of $\langle \psi | H | \psi \rangle$, where $H$ is the desired operator and $|\psi\rangle$ is the quantum state after executing a given kernel.


The `cudaq.observe()` method takes a kernel and its arguments as inputs, along with a `cudaq.operators.spin.SpinOperator`.

In [25]:
from cudaq import spin

operator = spin.z(0)
print(operator)  # prints: [1+0j] Z


@cudaq.kernel
def kernel():
    qubit = cudaq.qubit()
    h(qubit)

(1+0i) * Z0


In [26]:
result = cudaq.observe(kernel, operator)
print(result.expectation())  # prints: 0.0

0.0


Unlike sample, observe does not perform shots-based sampling by default (shots_count defaults to -1). In this mode the returned expectation value is computed analytically from the quantum state and is deterministic, equivalent to the expectation value in the limit of infinite shots. To produce an approximate expectation value from sampling instead, pass a positive integer as shots_count

In [27]:
result = cudaq.observe(kernel, operator, shots_count=1000)
print(result.expectation())  # prints non-zero value

0.03600000000000003
